##### Imports

In [ ]:
import numpy as np
import pandas as pd
import joblib
from pathlib import Path
from econml.dml import CausalForestDML

from gg570_d200.auxiliary_functions.forest_riesz_func import call_forestriesz, call_forestriesz_cross

In [134]:
root = Path.cwd().parent
data_path = root / "data"

In [135]:
np.random.seed(21)

In [136]:
df_scaled = pd.read_csv(data_path / "df_scaled.csv")

In [137]:
saved_joblib = joblib.load(data_path / "covariate_scaler.joblib")
scaler = saved_joblib['scaler']

In [138]:
df = pd.read_csv(data_path / "df.csv")

In [139]:
treatment_col = 'Training (last 3 months)'
outcome_col = 'Underemployment hours'
covariate_cols = [col for col in df_scaled.columns if col not in treatment_col and col not in outcome_col and col not in ['prop_scores']]

---

##### Key specification

In [140]:
call_forestriesz_cross(df_scaled, covariate_cols, treatment_col, outcome_col, methods=['dr', 'plugin'])

{'dr': {'est': -0.161563288559371,
  'low': -1.3453787351097957,
  'high': 1.0222521579910537},
 'plugin': {'est': -0.03872457663475174,
  'low': -0.4595074374118259,
  'high': 0.3820582841423225}}

---

##### Alternative specifications

###### ForestRiesz alternatives

In [141]:
call_forestriesz(df_scaled, covariate_cols, treatment_col, outcome_col, methods=['dr', 'plugin'])

{'dr': {'est': -0.23205636941886748,
  'low': -0.7955847974027555,
  'high': 0.33147205856502054},
 'plugin': {'est': -0.0367478621002449,
  'low': -0.296532478464972,
  'high': 0.22303675426448222}}

In [142]:
call_forestriesz_cross(df, covariate_cols, treatment_col, outcome_col, methods=['dr', 'plugin'])

{'dr': {'est': -0.1701567104620191,
  'low': -1.3543937866511178,
  'high': 1.0140803657270794},
 'plugin': {'est': -0.03936654703911537,
  'low': -0.460119042457307,
  'high': 0.3813859483790763}}

In [143]:
call_forestriesz(df, covariate_cols, treatment_col, outcome_col, methods=['dr', 'plugin'])

{'dr': {'est': -0.2376687525240957,
  'low': -0.8011547535109081,
  'high': 0.32581724846271676},
 'plugin': {'est': -0.037142018151062196,
  'low': -0.2968813883836602,
  'high': 0.22259735208153578}}

###### CausalForestDML

In [144]:
causal_dml = CausalForestDML(
    discrete_treatment=True,
    criterion='mse',
    n_estimators=100,
    min_samples_leaf=2,
    min_var_fraction_leaf=0.001,
    min_var_leaf_on_val=True,
    min_impurity_decrease=0.01,
    #max_samples=.8,
    max_depth=None,
    inference=True, #inference=False
    subforest_size=2, #subforest_size=1
    honest=True,
    verbose=0,
    n_jobs=-2,
    random_state=21,
    cv=3
)

In [145]:
causal_dml.fit(df_scaled[outcome_col], df_scaled[treatment_col], X=df_scaled[covariate_cols])
causal_dml.ate_, causal_dml.ate_stderr_

(array([-0.03379913]), array([0.42109106]))

###### Heterogeneity in ATEs

In [146]:
heterogeneity_importance = causal_dml.feature_importances_
max_id = int(np.argmax(heterogeneity_importance))
max_col = covariate_cols[max_id]
df_scaled[[max_col]].head()

,AGE
0,0.197454
1,-0.247202
2,0.997835
3,-1.136514
4,-1.225446


In [158]:
max_col_mean = scaler.mean_[max_id]
max_col_std = scaler.scale_[max_id]

df_scaled[f"{max_col}_original"] = df_scaled[max_col] * max_col_std + max_col_mean

DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`


In [162]:
df_scaled[f"{max_col}_original"].value_counts()

AGE_original
44.0    78
53.0    69
36.0    66
49.0    66
35.0    63
52.0    62
37.0    59
45.0    59
46.0    58
29.0    55
34.0    55
41.0    55
38.0    54
31.0    54
39.0    53
51.0    53
42.0    53
40.0    53
43.0    51
48.0    50
50.0    49
54.0    49
56.0    48
57.0    47
32.0    46
33.0    45
58.0    45
30.0    45
47.0    43
27.0    41
59.0    40
55.0    38
26.0    36
28.0    35
25.0    30
24.0    25
20.0    22
22.0    21
23.0    19
61.0    19
63.0    19
19.0    18
64.0    17
60.0    15
18.0    14
62.0    13
21.0    11
17.0     4
Name: count, dtype: int64

In [ ]:
group_mask = df_scaled['AGE'] > 0.2   # example subgroup
X_group = df_scaled.loc[group_mask, covariate_cols]

Rows in subgroup: 859


In [156]:
ate = causal_dml.ate(X=X_group)
lb, ub = causal_dml.ate_interval(X=X_group, alpha=0.05)

ate, (lb, ub)

(0.01629272445106373, (-5.058205454485087, 5.090790903387212))

In [132]:
# Heterogeneous effects by age (Option 2: representative person approach)
# Group by age, average covariates, predict effect for "average person" at each age

# Find age column (adjust if your column name is different)
age_col = max_col  # Change this to your actual age column name if different

# Ensure df_scaled has age column in original scale
if age_col not in df_scaled.columns:
    df_scaled[age_col] = df[age_col].values

# For each age group, get effects by averaging covariates
age_effects = []

for age in sorted(df_scaled[age_col].unique()):
    age_subset = df_scaled[df_scaled[age_col] == age]
    
    # Average the COVARIATES for this age group
    avg_covariates = age_subset[covariate_cols].mean().values.reshape(1, -1)
    
    # Predict effect at "average person" of this age
    effect = causal_dml.effect(avg_covariates)[0]
    ci_lower, ci_upper = causal_dml.effect_interval(avg_covariates)
    
    age_effects.append({
        'age': age,
        'n_obs': len(age_subset),
        'treatment_effect': effect,
        'conf_int_lower': ci_lower[0],
        'conf_int_upper': ci_upper[0]
    })

age_heterogeneous = pd.DataFrame(age_effects)
print(age_heterogeneous)

,merged_group,n_obs,gate,gate_sd
0,"[-2.203689, -1.847964]",69,-0.264164,0.904310
1,"[-1.759033, -1.403308]",131,-0.744473,1.027176
2,"[-1.314377, -0.958652]",230,-0.293248,0.884283
3,"[-0.869721, -0.513996]",275,0.068667,0.912916
4,"[-0.425065, -0.069340]",274,-0.046067,0.957592
5,"[0.019591, 0.375316]",299,-0.058742,0.969794
6,"[0.464247, 0.819972]",261,-0.091234,0.841836
7,"[0.908903, 1.264628]",266,-0.035498,0.947845
8,"[1.353559, 1.709284]",166,0.094010,1.219345
9,"[1.798216, 1.976078]",49,0.879654,2.519475


---

In [77]:
# Feature importance for heterogeneity in CausalForestDML
print("=" * 60)
print("VARIABLES DRIVING HETEROGENEITY")
print("=" * 60)

# Get feature importances from the causal forest
# These show which variables split most in the treatment effect forest
feature_importances = causal_dml.model_cate.feature_importances_

print(f"Feature importances shape: {feature_importances.shape}")
print(f"Feature importances type: {type(feature_importances)}")

# Flatten if needed
if feature_importances.ndim > 1:
    feature_importances = feature_importances.flatten()

# Get covariate names
feature_names = covariate_cols

# Match lengths
if len(feature_importances) != len(feature_names):
    print(f"⚠️  Mismatch: {len(feature_importances)} importances vs {len(feature_names)} features")
    # Trim to match
    min_len = min(len(feature_importances), len(feature_names))
    feature_importances = feature_importances[:min_len]
    feature_names = feature_names[:min_len]

# Create importance dataframe
importance_df = pd.DataFrame({
    'feature': feature_names,
    'importance': feature_importances
}).sort_values('importance', ascending=False)

print("\nTop 20 variables driving treatment effect heterogeneity:")
print(importance_df.head(20).to_string(index=False))

# Statistical summary
print(f"\nTotal features: {len(feature_importances)}")
print(f"Sum of importances: {feature_importances.sum():.4f}")
print(f"Top 5 account for: {importance_df.head(5)['importance'].sum():.4f} ({importance_df.head(5)['importance'].sum()/feature_importances.sum():.1%})")

# Identify if heterogeneity is driven by few vars or many
high_importance = (importance_df['importance'] > importance_df['importance'].quantile(0.75)).sum()
print(f"\nVariables in top quartile: {high_importance}")

if importance_df.iloc[0]['importance'] > importance_df['importance'].median() * 2:
    print("✓ Heterogeneity driven by FEW key variables")
else:
    print("✗ Heterogeneity spread across MANY variables")


VARIABLES DRIVING HETEROGENEITY
Feature importances shape: (1, 305)
Feature importances type: <class 'numpy.ndarray'>

Top 20 variables driving treatment effect heterogeneity:
                       feature  importance
                           AGE    0.160929
                        REFWKD    0.044657
         MARSTA_bin_(3.0, 4.0]    0.039009
                      HIQUAL22    0.033999
       LKWFWM_bin_(12.2, 15.0]    0.032238
                      HITQUA15    0.031588
         REFWKM_bin_(7.6, 9.8]    0.031029
         LKWFWM_bin_(6.6, 9.4]    0.025662
       WRKING_bin_(0.999, 1.5]    0.025407
       MARSTA_bin_(0.995, 2.0]    0.021414
        HRPID_bin_(0.999, 1.5]    0.020358
         WRKING_bin_(1.5, 2.0]    0.017840
         GOVTOF_bin_(6.4, 8.2]    0.017557
        NSECM20_bin_(6.6, 9.4]    0.016986
     APPR12_bin_(0.998, 1.667]    0.016882
          SATIS_bin_(4.0, 6.0]    0.016717
CRYOX7_EUL_Main_bin_(1.8, 2.6]    0.014497
       NATIDB11_bin_(0.5, 1.0]    0.014467
       